# Occupation experiment readiness
## tl;dr
The current v7 files require new resume-grouped splits and richer relevance judgments for occupation ranking. The Indian data can support a smaller occupation-stratified classification pilot. The shared projection can accept new feature vectors, but fresh text encoding is currently unavailable because the local MPNet model files are missing.
## Context & Methods
This notebook recomputes the source audit without modifying input data. Positive means top-level label equals 1. Identity uses normalized content hashes; it is not verified person identity. Feasibility screens require 25 positive training resumes and 10 positive test resumes absent from training and validation. These thresholds are not power calculations.


In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'scripts/audit_occupation_task_readiness.py').exists():
    root = root.parent
assert (root / 'scripts/audit_occupation_task_readiness.py').exists()
sys.path.insert(0, str(root))
from scripts.audit_occupation_task_readiness import audit
report, occupation_support = audit()


## Data and Results
Source files and their hashes are included in the source audit. The table below counts exact resume-content overlap and screens target support. Raw resumes are never displayed.


In [2]:
print('Dataset | Test rows | Test resumes | In training | Targets passing independent-support screen')
screen = 'train_positive_resumes>=25;independent_test_positive_resumes>=10'
for name, detail in report['datasets'].items():
    test = detail['splits']['test']
    overlap = detail['overlap']['train__test']['resume_contents']
    targets = detail['target_feasibility_screen'][screen]
    print(f'{name} | {test["rows"]} | {test["unique_resume_contents"]} | {overlap} | {targets}')


Dataset | Test rows | Test resumes | In training | Targets passing independent-support screen
career_v7 | 800 | 381 | 379 | 0
indian | 889 | 885 | 48 | 21
alitianchi_clean | 1401 | 817 | 786 | 0


### Model boundary check
The saved check distinguishes projection execution from fresh occupation text scoring. Finite outputs from zero feature vectors do not establish zero-shot accuracy.


In [3]:
model_check = json.loads((root / 'results/occupation_task_readiness/unseen_scoring_check.json').read_text())
print('Checkpoint loaded:', model_check.get('checkpoint_loaded', False))
print('New-text scoring completed:', model_check.get('new_text_scoring_completed', False))
print('Status:', model_check['status'])


Checkpoint loaded: True
New-text scoring completed: False
Status: checkpoint_and_projection_verified; text_encoding_unavailable


## Takeaways
Audit occupation mappings, deduplicate and group queries before splitting, and distinguish pair classification from occupation recommendation. Selected-target supervision and test candidate judgments must be defined before running learning curves. See docs/OCCUPATION_TASK_READINESS.md for findings and the scope of each conclusion.
